# 05 - Geography, boundaries and the IEC Atlas

This notebook does three things for the master table:

1. Municipal boundaries: the map for the dashboard, the area of each municipality, and a list of neighbours.
2. A 2011 to today mapping. Boundaries were redrawn in 2016, so the 2011 results do not line up with today's
   municipality codes. We work out the mapping from the maps themselves.
3. IEC Atlas data: registration activity per municipality (new registrations and people who moved), which is not
   published anywhere else.

Sources: the Municipal Demarcation Board (MDB) Spatial Knowledge Hub for boundaries, and the IEC local election
Atlas (atlas.elections.org.za/lgeatlas/). Both are in `docs/sources.csv`.

## 0. Setup

Loads the shared settings: paths, folders and the random seed.

In [ ]:
import sys
from pathlib import Path

REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "src" / "config.py").exists())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src import config
from src.config import RAW, INTERIM, PROCESSED, FIGURES, SEED

import json, time, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
print(config.describe())

## 1. Municipal boundaries

We use the MDB's "Local Municipalities 2026" layer, downloaded as GeoJSON. It already includes the eight metros
(`CATEGORY` is A for metros and B for local municipalities). District municipalities are not in it, which is what we
want: every voter lives in a local municipality or a metro, so districts would count people twice.

In [ ]:
gdf = gpd.read_file(RAW / "boundaries" / "MDB_L_ocal_Municipalities_2026.geojson")

print(gdf.shape, gdf.crs)
print(gdf.columns.tolist())
gdf.head(3)

Result: 213 rows. The municipality code is in `CAT_B` (for example `BUF` or `EC101`), which matches the IEC files.
`MUNICNAME` is the name, `CATEGORY` says metro or local, and `Area_sqkm` is the MDB's area.

We rename these to our standard names (`muni_code`, `muni_name`, and so on) and drop the columns we do not need.

In [ ]:
gdf = gdf.rename(columns={"CAT_B": "muni_code", "MUNICNAME": "muni_name",
                          "PROVINCE": "province", "CATEGORY": "muni_category",
                          "DISTRICT": "district_code", "Area_sqkm": "area_km2"})
gdf = gdf[["muni_code", "muni_name", "province", "muni_category",
           "district_code", "area_km2", "geometry"]]

print(gdf.shape)
gdf.head(3)

### Checks

Four things must be true before we use this layer.

In [ ]:
print("duplicate codes :", gdf["muni_code"].duplicated().sum())
print("category counts :", gdf["muni_category"].value_counts().to_dict())
print(f"total area      : {gdf['area_km2'].sum():,.0f} km2   (South Africa ~1 220 000 km2)")

# cross-check MDB's area against our own calculation
gdf["area_km2_calc"] = gdf.to_crs(epsg=3857).area / 1e6
print("\nMDB vs computed area (Web Mercator distorts area at SA latitudes,")
print("so we keep MDB's figure as authoritative):")
print(gdf[["area_km2", "area_km2_calc"]].describe().round(0))

ax = gdf.plot(figsize=(7, 7), edgecolor="white", linewidth=0.2, color="#4a5059")
ax.set_axis_off(); ax.set_title("213 municipalities, MDB 2026 boundaries")
plt.show()

All four checks pass: no duplicate codes, 8 metros and 205 local municipalities, the total area matches South
Africa, and the map shows the whole country.

We also calculated the area ourselves, but the map projection we used stretches areas in South Africa, so we use
the MDB's own area figure.

## 2. Neighbours

Which municipalities share a border. We use this for the "neighbours' turnout" feature: the average turnout of the
municipalities around each one.

Low turnout tends to cluster in regions, so a municipality with low-turnout neighbours is more likely to have low
turnout too. This feature lets the model use that pattern.

Each border is listed twice (A to B and B to A), so every municipality gets its full list of neighbours.

In [ ]:
nb = gpd.sjoin(gdf[["muni_code", "geometry"]],
               gdf[["muni_code", "geometry"]],
               predicate="touches")[["muni_code_left", "muni_code_right"]]
nb = nb[nb.muni_code_left != nb.muni_code_right].drop_duplicates()

print(len(nb), "directed border pairs")
print(nb.groupby("muni_code_left").size().describe().round(2).to_string())

nb.to_csv(INTERIM / "neighbours.csv", index=False)

Result: 1,100 pairs (about 550 borders). On average a municipality has 5 neighbours (between 1 and 10). None has
zero, which also tells us the maps have no broken shapes.

## 3. The IEC Atlas

The IEC has an online Atlas of results (atlas.elections.org.za/lgeatlas/). It looks like a map with no download
button. But when we looked at the page's network requests, we found that its data comes from an open map server
(GeoServer). So we can download every layer behind the map as data.

- The list of layers is in `lgeatlas/Scripts/report_layers.js` (it is JSON, even though the name ends in .js).
- The data itself comes from `https://atlas.elections.org.za:8443/geoserver/wfs`.

This gives us registration activity per municipality, which is not on the registration dashboard or in the results
files.

### 3.1 The list of layers

In [ ]:
js = requests.get("https://atlas.elections.org.za/lgeatlas//Scripts/report_layers.js",
                  timeout=60).text
cat = json.loads(js[js.index("{"):])

rows = [{"theme": h["label"], "group": g["label"], "year": l["label"],
         "layer": l["lyrurl"], "key": l.get("titlekey"),
         "cols": [c["name"] for c in l.get("columns", [])]}
        for h in cat["grouplayers"] for g in h["groups"] for l in g["layers"]]

layers = pd.DataFrame(rows)
layers.to_csv(INTERIM / "atlas_layers.csv", index=False)

print(len(layers), "layers")
layers["theme"].value_counts()

There are 181 layers, about voting districts, registration, voting stations and results, for elections from 2000
to 2021.

Three kinds are useful for us:
- total registration activity per municipality (2011 and 2016): new registrations, people re-registering, and
  people who moved to another voting district
- how many new voting districts each municipality got
- how densely people are registered in each voting district (2011, 2016, 2021)

### 3.2 Downloading one layer

In [ ]:
GS = "https://atlas.elections.org.za:8443/geoserver/wfs"

def get_layer(type_name, max_features=None, timeout=300):
    """Request one Atlas layer as GeoJSON. Pass max_features to inspect a schema cheaply."""
    params = {"service": "WFS", "version": "1.0.0", "request": "GetFeature",
              "typeName": type_name, "outputFormat": "application/json"}
    if max_features:
        params["maxFeatures"] = max_features
    r = requests.get(GS, params=params, timeout=timeout)
    r.raise_for_status()
    return r

(RAW / "atlas").mkdir(exist_ok=True)

### 3.3 Registration activity in 2016

Layer `LGEAtlas:Total registration combined 2016`. We keep the numbers and drop the map shapes, because we already
have boundaries from the MDB.

How the columns are named: plain names are registration weekend 1, names starting with `X` are weekend 2, and names
ending in `3` are both weekends together.

- `REGPOP`, `XPOP`, `POP3`: registered population
- `NEW_REG`, `XNEWREG`, `NEWREG3`: new registrations (first time on the roll)
- `VD_MOVE`, `XVDMOVE`, `VDM3`: people who re-registered in a different voting district (they moved)
- `REREG`, `REREG2`, `REREG3`: people who re-registered in the same voting district
- `TOTALW1`, `XTOTAL`, `TOTAL3`: total activity

In [ ]:
r = get_layer("LGEAtlas:Total registration combined 2016")
(RAW / "atlas" / "registration_activity_2016.json").write_text(r.text, encoding="utf-8")
print(r.headers.get("content-type"), f"{len(r.content)/1e6:.1f} MB")

ra = gpd.read_file(RAW / "atlas" / "registration_activity_2016.json")
print(ra.shape)
print(ra.columns.tolist())

In [ ]:
ra = ra.rename(columns={"MUNICCODE": "muni_code"})
keep = ["muni_code", "REGPOP", "NEW_REG", "VD_MOVE", "REREG", "TOTALW1",
        "XPOP", "XNEWREG", "XVDMOVE", "REREG2", "XTOTAL",
        "POP3", "NEWREG3", "VDM3", "REREG3", "TOTAL3"]
ra2016 = pd.DataFrame(ra[keep]).copy()

# same codes as the 2026 boundaries, so 2016 needs no mapping
assert set(ra2016["muni_code"]) == set(gdf["muni_code"])

ra2016.to_csv(INTERIM / "atlas_registration_activity_2016.csv", index=False)
print(ra2016.shape)
ra2016.head(3)

All 213 municipality codes in the 2016 Atlas layer are the same as in the 2026 MDB boundaries. So boundaries did not
change from 2016 to 2021 to 2026, and only 2011 needs a mapping.

### 3.4 Registration activity in 2011

Layer `LGEAtlas:Total_registration_Activity_(Combined) 2011`. This one uses the 2011 boundaries, so it has 234
municipalities and some of its codes are not current.

In [ ]:
time.sleep(3)   # short pause between requests

r11 = get_layer("LGEAtlas:Total_registration_Activity_(Combined) 2011")
(RAW / "atlas" / "registration_activity_2011.json").write_text(r11.text, encoding="utf-8")

ra11 = gpd.read_file(RAW / "atlas" / "registration_activity_2011.json")
print(ra11.shape)
print(ra11.columns.tolist())

# CODE holds the municipality code; SMUNICIPAL holds "CODE - NAME"
for c in ["CODE", "SMUNICIPAL"]:
    print(f"{c:12s}", ra11[c].head(3).tolist())

In [ ]:
ra2011 = pd.DataFrame(ra11[["CODE", "SMUNICIPAL",
                            "TOTAL_ACTI", "TOTAL_ACT1", "TOTALACTIV"]]).copy()
ra2011 = ra2011.rename(columns={"CODE": "muni_code_2011",
                                "SMUNICIPAL": "muni_name_2011"})
ra2011.to_csv(INTERIM / "atlas_registration_activity_2011.csv", index=False)
print(ra2011.shape)
ra2011.head(3)

We call the code column `muni_code_2011` on purpose. These are 2011 codes, and joining them straight to today's data
would be wrong (see section 4).

## 4. Mapping 2011 municipalities to today's

### Why we need it

Boundaries were redrawn for the 2016 election and several municipalities were merged. There were 234 municipalities
in 2011 and there are 213 today.

Some codes were even reused for a different area. For example, `EC101`:
- in 2011 it was Camdeboo, with 23,164 registered voters
- from 2016 it is Dr Beyers Naude (Camdeboo + Ikwezi + Baviaans), with 40,887

Joining on the code alone would make it look like registration grew by 72%, when really two municipalities were
added.

### How we do it

We lay the 2011 map over today's map and measure how much of each old municipality's area now falls inside each
current one. That share is the weight we use to split the 2011 numbers. It is quick, and anyone can re-run it.

In [ ]:
# 2011 boundaries come from the Atlas layer we already pulled
g11 = ra11.to_crs(epsg=3857)[["CODE", "SMUNICIPAL", "geometry"]]
cur = gdf.to_crs(epsg=3857)[["muni_code", "muni_name", "geometry"]]

ov = gpd.overlay(g11, cur, how="intersection")
ov["overlap_km2"] = ov.area / 1e6
ov["share"] = ov["overlap_km2"] / ov.groupby("CODE")["overlap_km2"].transform("sum")

print(len(ov), "raw intersection pieces")

### Small overlaps

Moving a border by a few metres creates tiny overlaps that do not mean anything. We drop overlaps smaller than 1% of
the old municipality's area.

In [ ]:
multi = ov[ov["share"] >= 0.01].groupby("CODE").size()
split_codes = multi[multi > 1].index.tolist()

sub = ov[ov["CODE"].isin(split_codes) & (ov["share"] >= 0.01)]
print(sub.sort_values(["CODE", "share"], ascending=[True, False])
        [["CODE", "SMUNICIPAL", "muni_code", "muni_name", "share"]]
        .to_string(index=False))

Nine old municipalities are split between more than one current municipality. Five of them are real splits:

- FS203 Ngwathe: Ngwathe 0.51, Metsimaholo 0.49
- LIM342 Mutale: Musina 0.39, Collins Chabane 0.32, Thulamela 0.29
- KZN283 Ntambanana: Mthonjaneni 0.54, uMhlathuze 0.41, Mfolozi 0.05
- LIM343 Thulamela: Collins Chabane 0.74, Thulamela 0.26
- LIM352 Aganang: Polokwane 0.70, Blouberg 0.15, Molemole 0.15

The other four go mostly (77% to 89%) to one municipality. Because we split by share anyway, all of them are handled
the same way.

In [ ]:
cw = ov[ov["share"] >= 0.01].copy()
cw = cw.rename(columns={"CODE": "muni_code_2011", "SMUNICIPAL": "name_2011",
                        "muni_code": "muni_code_current"})
cw = cw[["muni_code_2011", "name_2011", "muni_code_current", "muni_name", "share"]]
cw.to_csv(INTERIM / "muni_crosswalk_2011.csv", index=False)
print(cw.shape)
cw.head(5)

### Check the mapping

In [ ]:
print(cw["muni_code_2011"].nunique(), "of 234 old codes covered")

s = cw.groupby("muni_code_2011")["share"].sum()
print("\nshares per old municipality (should sum to ~1):")
print(s.describe().round(4).to_string())
print("\nany losing more than 2% to the sliver threshold:", (s < 0.98).sum())

All 234 old codes are covered. The shares add up to 0.999 on average, and at least 0.986, so dropping the tiny
overlaps loses very little.

### How to use it

```python
# split the counts by share, then add them up per current municipality
m = df_2011.merge(cw, left_on="muni_code", right_on="muni_code_2011", how="left")
for c in ["registered", "votes_cast", "valid_votes", "spoilt"]:
    m[c] = m[c] * m["share"]
out = m.groupby("muni_code_current", as_index=False)[counts].sum()

# work out turnout again after adding up (never split a percentage)
out["turnout"] = out["votes_cast"] / out["registered"]
```

Split the counts, add them up, and only then divide. Splitting percentages gives wrong numbers that still look fine.

With this mapping the 2011 national totals hardly change (23,654,347 to 23,623,818 registered voters, 0.13% lost to
tiny overlaps) and national turnout stays 0.5745. `EC101` in 2011 becomes 38,211 registered voters (Camdeboo +
Ikwezi + Baviaans), which fits well with the 40,887 in 2016.

## 5. Save

In [ ]:
gdf.to_file(INTERIM / "boundaries.gpkg", driver="GPKG")

for f in sorted(INTERIM.glob("*")):
    if f.is_file() and f.name != ".gitkeep":
        print(f"{f.name:45s} {f.stat().st_size/1024:8.1f} KB")

## 6. Summary and limitations

What we found:
1. The IEC Atlas data can be downloaded from an open map server, so its 181 layers can be used as data.
2. Registration activity (new registrations, re-registrations and moves) is only published in the Atlas.
3. Municipality codes are the same in 2016 and 2026, so only 2011 needs a mapping.
4. The 2011 mapping covers all 234 old codes, and the five real splits are shared out by area.

Limitations:
- We split by area, not by population. Where people are not spread evenly, the 2011 numbers are approximate. We
  cannot do better without population figures per voting district for 2011, which are not published.
- Overlaps under 1% are dropped. This loses 0.13% of the 2011 registered voters.
- The Atlas only has registration activity for 2011 and 2016, so we cannot use it to forecast 2026. We use it only
  for exploring the data.
- We use the MDB's area, not our own calculation, because our map projection stretches areas.

Where the files are used:
- `boundaries.gpkg`: the dashboard map, and area and density in the master table
- `neighbours.csv`: neighbours' turnout in the master table
- `muni_crosswalk_2011.csv`: the 2011 results in `06_master`
- `atlas_registration_activity_*.csv`: exploring the data